## Árvore de Decisão

A [Árvore de Decisão](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html) é um modelo baseado em regras, que realiza divisões sucessivas nos dados com o objetivo de separar as classes.

Cada divisão é feita com base em uma variável e um ponto de corte, formando uma estrutura semelhante a uma árvore.

Esse modelo é interpretável e capaz de capturar relações não lineares entre as variáveis.

### Hiperparâmetros utilizados

- **max_depth**: define a profundidade máxima da árvore.
  - Valores menores → modelo mais simples (menos overfitting)
  - Valor `None` → árvore cresce livremente

- **min_samples_split**: número mínimo de amostras necessárias para dividir um nó.
  - Valores maiores tornam o modelo mais conservador

- **min_samples_leaf**: número mínimo de amostras em uma folha.
  - Evita que a árvore crie divisões muito específicas

- **criterion**: métrica utilizada para avaliar a qualidade da divisão.
  - `gini`: índice de Gini (default)
  - `entropy`: ganho de informação

### Estratégia

Foi utilizado o GridSearchCV com validação cruzada (5 folds) para ajustar os hiperparâmetros e reduzir o risco de overfitting.

In [1]:
import pandas as pd
import sys
import os

# add raiz do projeto
sys.path.append(os.path.abspath(".."))

from sklearn.model_selection import GridSearchCV
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score

from preprocessing.main_preprocessing import load_and_preprocess
from utils.experiment_logger import save_results


## BASELINE

In [2]:
scenarios = [
    "sem_submodalidade",
    "submodalidade_agrupada",
    "submodalidade_engineered"
]

smote_options = [False, True]

results = []


for scenario in scenarios:
    for use_smote in smote_options:

        print(f"\n🔎 Scenario: {scenario} | SMOTE: {use_smote}")

        X_train, X_test, y_train, y_test = load_and_preprocess(
            "../predictive_models/scrdata_202505.csv",
            scenario=scenario,
            use_smote=use_smote
        )

        steps = [
            ('dt', DecisionTreeClassifier(random_state=42))
                    ]

        if use_smote:
            steps.insert(1 if not steps or steps[0][0] != 'scaler' else 1, ('smote', SMOTE(random_state=42)))

        model = Pipeline(steps)

        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1]

        results.append({
            "model": "DecisionTree",
            "scenario": scenario,
            "smote": use_smote,
            "roc_auc": roc_auc_score(y_test, y_proba),
            "f1": f1_score(y_test, y_pred),
            "accuracy": accuracy_score(y_test, y_pred),
            "n_features": X_train.shape[1],
            "phase": "baseline",
            "timestamp": pd.Timestamp.now()
        })


save_results(results, file_path="../results/experiments.csv")

df_result = pd.DataFrame(results)

display(df_result)



🔎 Scenario: sem_submodalidade | SMOTE: False


NotFittedError: Pipeline is not fitted yet.

## GRIDSEARCH

In [ ]:
X_train, X_test, y_train, y_test = load_and_preprocess(
    "../predictive_models/scrdata_202505.csv",
    scenario="sem_submodalidade",
    use_smote=False
)


param_grid_dt = {
    "smote": [SMOTE(random_state=42), "passthrough"],
    "dt__max_depth": [10, 20, 30],
    "dt__min_samples_split": [2, 5, 10],
    "dt__min_samples_leaf": [1, 2, 4],
    "dt__criterion": ["gini", "entropy"]
}

grid_dt = GridSearchCV(
    estimator=Pipeline([
        ('smote', SMOTE(random_state=42)),
        ('dt', DecisionTreeClassifier(random_state=42))
    ]),
    param_grid=param_grid_dt,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1
)

grid_dt.fit(X_train, y_train)

print("Best parameters:", grid_dt.best_params_)
print("Best ROC AUC (CV):", grid_dt.best_score_)


#? BEST MODEL TEST EVALUATION

best_dt = grid_dt.best_estimator_

y_pred = best_dt.predict(X_test)
y_proba = best_dt.predict_proba(X_test)[:, 1]


#? TUNING (CV)

tuning_result = [{
    "model": "DecisionTree",
    "scenario": "sem_submodalidade",
    "smote": False,
    "phase": "tuning_cv",
    "roc_auc": grid_dt.best_score_,
    "f1": None,
    "accuracy": None,
    "best_params": str(grid_dt.best_params_),
    "timestamp": pd.Timestamp.now()
}]

final_result = [{
    "model": "DecisionTree",
    "scenario": "sem_submodalidade",
    "smote": False,
    "phase": "test",
    "roc_auc": roc_auc_score(y_test, y_proba),
    "f1": f1_score(y_test, y_pred),
    "accuracy": accuracy_score(y_test, y_pred),
    "best_params": str(grid_dt.best_params_),
    "timestamp": pd.Timestamp.now()
}]


save_results(tuning_result, file_path="../results/model_results.csv")
save_results(final_result, file_path="../results/model_results.csv")


df_result = pd.DataFrame(final_result)
display(df_result)


Best parameters: {'criterion': 'entropy', 'max_depth': 10, 'min_samples_leaf': 4, 'min_samples_split': 10}
Best ROC AUC (CV): 0.9113779414410601


,model,scenario,smote,phase,roc_auc,f1,accuracy,best_params,timestamp
0,DecisionTree,sem_submodalidade,False,test,0.910683,0.854567,0.830973,"{'criterion': 'entropy', 'max_depth': 10, 'min...",2026-04-09 21:57:58.046473
